# 低通滤波对方波脉冲的影响：频域与时域交互可视化

本 Notebook 通过滑块调节低通滤波器的截止频率，实时展示：
- 原始脉冲与滤波后脉冲的频谱对比
- 原始脉冲与滤波后脉冲的时域波形（展宽现象）

拖动滑块，观察截止频率降低时，高频分量被削减，时域脉冲如何逐渐展宽并产生拖尾。

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from ipywidgets import interact, FloatSlider

# ----- 设置中文字体，避免警告 -----
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'KaiTi', 'Arial Unicode MS']  # 按优先级尝试
plt.rcParams['axes.unicode_minus'] = False  # 正确显示负号
# ---------------------------------

# 参数设置
fs = 1000               # 采样率 (Hz)
T = 1.0                 # 总时间 (秒)
N = int(fs * T)         # 采样点数
t = np.linspace(0, T, N, endpoint=False)

# 生成一个矩形脉冲（模拟一个符号）
pulse_start = 0.2       # 起始时间 (秒)
pulse_end = 0.3         # 结束时间 (秒)
x = np.zeros(N)
x[(t >= pulse_start) & (t < pulse_end)] = 1.0

def plot_filtered(fc):
    # 设计二阶巴特沃斯低通滤波器
    b, a = signal.butter(2, fc/(fs/2), btype='low')
    y = signal.filtfilt(b, a, x)   # 零相位滤波，避免相位偏移
    
    # 计算频谱
    X = np.fft.fft(x)
    Y = np.fft.fft(y)
    freqs = np.fft.fftfreq(N, 1/fs)[:N//2]
    
    # 计算滤波器频率响应
    w, h = signal.freqz(b, a, worN=8000, fs=fs)
    
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 8))
    
    # 频域子图
    ax1.plot(freqs, np.abs(X[:N//2]), label='原始信号频谱', alpha=0.7)
    ax1.plot(freqs, np.abs(Y[:N//2]), label='滤波后信号频谱', alpha=0.7)
    scale = np.max(np.abs(X[:N//2])) / np.max(np.abs(h))
    ax1.plot(w, np.abs(h) * scale, 'k--', label='滤波器响应（缩放）')
    ax1.axvline(fc, color='r', linestyle=':', label=f'截止频率 fc = {fc} Hz')
    ax1.set_xlim(0, 200)
    ax1.set_xlabel('频率 (Hz)')
    ax1.set_ylabel('幅度')
    ax1.legend(loc='upper right')
    ax1.grid(True)
    ax1.set_title('频域：原始与滤波后频谱对比')
    
    # 时域子图
    ax2.plot(t, x, label='原始脉冲')
    ax2.plot(t, y, label='滤波后脉冲')
    ax2.set_xlim(0, 0.6)
    ax2.set_xlabel('时间 (秒)')
    ax2.set_ylabel('幅度')
    ax2.legend()
    ax2.grid(True)
    ax2.set_title('时域：脉冲展宽与拖尾')
    
    plt.tight_layout()
    plt.show()

fc_slider = FloatSlider(value=50, min=1, max=200, step=1, description='截止频率 (Hz)')
interact(plot_filtered, fc=fc_slider);

interactive(children=(FloatSlider(value=50.0, description='截止频率 (Hz)', max=200.0, min=1.0, step=1.0), Output()…

## 使用说明
1. 运行上面代码单元格，稍等片刻会显示滑块和图表。
2. 拖动滑块改变截止频率，图表将实时更新。
3. 观察当截止频率降低时：
   - 频域上，高于截止频率的分量被显著衰减；
   - 时域上，脉冲边沿变缓，宽度增加，出现明显的前导和拖尾（符号展宽）。

这正是低通滤波器导致码间干扰（ISI）的根本原因。